# Lab 01 — SQL no navegador (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite) — o DuckDB é instalado sob demanda e roda no navegador. Execute célula a célula (Shift+Enter).

Objetivo: praticar `SELECT`, `WHERE`, `ORDER BY` e `GROUP BY` sobre uma tabela de pedidos realista.

In [ ]:
# Setup: garante o duckdb (no navegador instala sob demanda)
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
import pandas as pd
print('duckdb', duckdb.__version__)

## 1. Os dados (uma tabela de pedidos)
O DuckDB consegue consultar um DataFrame do pandas pelo nome da variável.

In [ ]:
pedidos = pd.DataFrame([
    (1,'SP','eletronicos',1200.0,1),(2,'SP','livros',50.0,2),(3,'RJ','livros',30.0,1),
    (4,'MG','casa',80.0,3),(5,'SP','eletronicos',800.0,2),(6,'RJ','casa',150.0,4),
    (7,'SP','livros',45.0,1),(8,'MG','eletronicos',600.0,3),(9,'RJ','eletronicos',900.0,2),
    (10,'SP','casa',200.0,5),(11,'MG','livros',25.0,4),(12,'SP','eletronicos',1500.0,1),
    (13,'RJ','livros',60.0,5),(14,'MG','casa',120.0,3),(15,'SP','livros',40.0,2),
], columns=['id','estado','categoria','valor','cliente_id'])
pedidos.head()

## 2. SELECT + WHERE + ORDER BY

In [ ]:
duckdb.query('''
    SELECT id, categoria, valor
    FROM pedidos
    WHERE estado = 'SP' AND valor > 100
    ORDER BY valor DESC
''').to_df()

## 3. GROUP BY: receita por estado

In [ ]:
duckdb.query('''
    SELECT estado, COUNT(*) AS n, SUM(valor) AS receita
    FROM pedidos
    GROUP BY estado
    ORDER BY receita DESC
''').to_df()

## 4. Sua vez (mini-desafio)
Escreva uma query que traga a **receita por categoria** (colunas `categoria` e `receita`), da maior para a menor. Rode a verificação.

In [ ]:
resposta = duckdb.query('''
    SELECT categoria, SUM(valor) AS receita
    FROM pedidos
    GROUP BY categoria
    ORDER BY receita DESC
''').to_df()
resposta

In [ ]:
def verificar(df):
    try:
        assert list(df.columns) == ['categoria','receita'], 'Colunas devem ser categoria e receita.'
        top = df.iloc[0]
        assert top['categoria'] == 'eletronicos' and abs(float(top['receita'])-5000.0) < 1e-6, 'eletronicos deveria liderar com 5000.'
        print('\u2705 Correto! Agregou e ordenou certinho.')
    except AssertionError as e:
        print('\u274c', e)

verificar(resposta)